### csv 피처들만 남기기

In [32]:
import pandas as pd

for i in range(16,26):
    # 인코딩이 euc-kr로 되어있음
    telecom_df = pd.read_csv(f'../../data/raw/p{i}v32_KMP_csv.csv', encoding='euc-kr')
    telecom_df = telecom_df.rename(columns={
        f'p{i}pid': 'id',
        f'p{i}gender': 'gender',
        f'p{i}byear': 'birth_year',
        f'p{i}school': 'school',
        f'p{i}mar':'mar',
        f'p{i}income1':'income',
        f'p{i}job1':'job',
        f'p{i}area':'region',
        f'p{i}c01001':'phone_usage_per_m', 
        f'p{i}a03008':'telecom'
    })
    telecom_df['year'] = f'20{i}'
    # 나이 컬럼 추가
    telecom_df['age'] = telecom_df['year'].astype(int) - telecom_df['birth_year']
    telecom_df = telecom_df[['id', 'gender', 'age', 'year', 'school', 'mar', 'income', 'job', 'region', 'phone_usage_per_m', 'telecom']]
    
    # 저장할 때 utf-8로 변환
    telecom_df.to_csv(f"../../data/processed/p{i}v32_KMP_csv.csv", index=False, encoding="utf-8")

C:\Users\Playdata\AppData\Local\Temp\ipykernel_4864\2140177334.py:5: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  telecom_df = pd.read_csv(f'../../data/raw/p{i}v32_KMP_csv.csv', encoding='euc-kr')
C:\Users\Playdata\AppData\Local\Temp\ipykernel_4864\2140177334.py:5: DtypeWarning: Columns (4,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  telecom_df = pd.read_csv(f'../../data/raw/p{i}v32_KMP_csv.csv', encoding='euc-kr')
C:\Users\Playdata\AppData\Local\Temp\ipykernel_4864\2140177334.py:5: DtypeWarning: Columns (5,8,9,73) have mixed types. Specify dtype option on import or set low_memory=False.
  telecom_df = pd.read_csv(f'../../data/raw/p{i}v32_KMP_csv.csv', encoding='euc-kr')
C:\Users\Playdata\AppData\Local\Temp\ipykernel_4864\2140177334.py:5: DtypeWarning: Columns (5,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  telecom_df = pd.read_csv(f'../../data/raw/p{i}v32_KM

### csv 합치기

In [33]:
import glob

files = glob.glob("../../data/processed/*.csv")

df_list = [pd.read_csv(f) for f in files]
merged_df = pd.concat(df_list, ignore_index=True)

merged_df.to_csv("../../data/processed/merged_telecom.csv", index=False)

In [31]:
telecom2_df = pd.read_csv('../../data/processed/telecom.csv')
telecom2_df = telecom2_df[telecom2_df['year']==2017]
telecom3_df = pd.read_csv('../../data/processed/p16v32_KMP_csv.csv')

# 2에는 있는데 3에는 없는 ID의 개수
missing_count = telecom2_df[~telecom2_df['id'].isin(telecom3_df['id'])].shape[0]

print(f"telecom2 전체 데이터: {len(telecom2_df)}개")
print(f"telecom3에 없는 ID 개수: {missing_count}개")
print(f"누락 비율: {(missing_count / len(telecom2_df)) * 100:.2f}%")

telecom2 전체 데이터: 3819개
telecom3에 없는 ID 개수: 125개
누락 비율: 3.27%


### csv 결측치 제거

In [34]:
telecom_df = pd.read_csv('../../data/processed/merged_telecom.csv')
telecom_df = telecom_df[(telecom_df != " ").all(axis=1)]
telecom_df_count = telecom_df.groupby('id')['id'].transform('count')
telecom_df = telecom_df[telecom_df_count>=10]
telecom_df.to_csv("../../data/processed/merged_telecom.csv", index=False)

In [55]:
# 같은 pid값에 9개씩 있는지 확인
pd.set_option('display.max_rows', None)
telecom_df['id'].value_counts()

id
10002       9
20001       9
30001       9
30002       9
30003       9
50002       9
60001       9
60003       9
100001      9
100002      9
130001      9
160001      9
160002      9
180001      9
220001      9
220002      9
260001      9
260002      9
270001      9
270002      9
290002      9
300001      9
300002      9
300003      9
310001      9
310003      9
350001      9
350003      9
360001      9
360002      9
380002      9
390001      9
390002      9
390003      9
390004      9
400002      9
410002      9
420001      9
420002      9
430001      9
430002      9
440001      9
440003      9
440004      9
460001      9
460002      9
470001      9
470003      9
490001      9
490003      9
500001      9
520001      9
520002      9
530003      9
540002      9
550002      9
550003      9
560001      9
560002      9
570001      9
570002      9
590001      9
600002      9
600005      9
610001      9
620001      9
630001      9
630002      9
640001      9
660002      9
670001      9
670

### 통신사 이탈 여부 컬럼 추가
- 작년과 비교했을 때 통신사가 달라졌으면 이탈로 간주
- 이탈했을 경우 1 아닌 경우 0
- 2017은 비교 대상이 없으니 0

In [37]:
telecom_df = pd.read_csv('../../data/processed/merged_telecom.csv')
pd.set_option('display.max_rows', None)

# 아이디,년도 별로 정렬
telecom_df = telecom_df.sort_values(['id', 'year'])

# 아이디별로 묶어 만약 이전 row와 값이 1
telecom_df['telecom_change'] = telecom_df.groupby('id')['telecom'].shift(1)
telecom_df['telecom_change_yn'] = (telecom_df['telecom_change'] != telecom_df['telecom']).astype(int)

telecom_df['telecom_change'] = telecom_df['telecom_change'].astype('Int64')
telecom_df = telecom_df.drop('telecom_change', axis=1)
# 2017은 이전 값이 없으니 0
telecom_df.loc[telecom_df['year']==2016, 'telecom_change_yn'] = 0
#telecom_df.loc[telecom_df['year']==2017, 'telecom_change'] = telecom_df['telecom']
telecom_df.to_csv("../../data/processed/telecom.csv", index=False)

In [ ]:

telecom_df = telecom_df[telecom_df['year'] != 2016]
telecom_df.to_csv("../../data/processed/telecom.csv", index=False)

In [ ]:
telecom = pd.read_csv('../../data/processed/telecom.csv')
telecom2016 = pd.read_csv('../../data/processed/telecom2016.csv')

telecom2016 = pd.merge(telecom2016, 
                    telecom[['id', 'year', 'mobile_bundle']], 
                    on=['id', 'year'], 
                    how='left')
telecom2016.to_csv("../../data/processed/telecom_f.csv", index=False)

### job, gender 컬럼 0, 1로 변환

In [65]:
telecom_df = pd.read_csv('../../data/processed/telecom_f.csv')
telecom_df['gender'] = telecom_df['gender'].replace({1: 0, 2: 1})
telecom_df['job'] = telecom_df['job'].replace({1: 1, 2: 0})
telecom_df['mobile_bundle'] = telecom_df['mobile_bundle'].replace({1: 1, 2: 0})
telecom_df.to_csv("../../data/processed/telecom_f.csv", index=False)

### 인코딩

In [66]:
import pandas as pd
telecom_df = pd.read_csv('../../data/processed/telecom_f.csv')
telecom_df['school'] = telecom_df['school']-1
telecom_df['income'] = telecom_df['income']-1
df_encoded = pd.get_dummies(telecom_df, columns=['mar', 'region', 'telecom'], dtype='int')

df_encoded.to_csv("../../data/processed/telecom_encoding.csv", index=False)

In [67]:
telecom_df = pd.read_csv("../../data/processed/telecom_encoding.csv")
telecom_df = telecom_df.rename(columns={
    'region_1': 'seoul',
    'region_2': 'busan',
    'region_3': 'daegu',
    'region_4': 'incheon',
    'region_5': 'gwangju',
    'region_6': 'daejeon',
    'region_7': 'ulsan',
    'region_8': 'gyeonggi',
    'region_9': 'gangwon',
    'region_10': 'chungbuk',
    'region_11': 'chungnam',
    'region_12': 'jeonbuk',
    'region_13': 'jeonnam',
    'region_14': 'gyeongbuk',
    'region_15': 'gyeongnam',
    'region_16': 'jeju',
    'region_17': 'sejong',
    'telecom_1': 'skt',
    'telecom_2': 'kt',
    'telecom_3': 'lgu',
    'telecom_4': 'mvno',
})

# 저장할 때 utf-8로 변환
telecom_df.to_csv("../../data/processed/telecom_encoding.csv", index=False)